# ViCTSD — Baseline CE trên toàn bộ prompt và response

Baseline **cross-entropy thuần**: tính loss trên system prompt, ví dụ one-shot, nội dung đầu vào và JSON trả lời. Không completion masking, không class alpha và không focal modulation. Chỉ padding bị gán `-100`; EOS thật vẫn được học, kể cả khi tokenizer dùng chung ID cho padding và EOS. Causal shift loại token đầu khỏi các target dự đoán như mô hình ngôn ngữ thông thường.

$$L_{\mathrm{CE}}=\frac{1}{|M|}\sum_{t\in M}-\log p(y_t\mid y_{<t}),$$

với `M` là mọi target không phải padding sau causal shift, thuộc cả prompt và response. Không chuẩn hóa riêng theo số token nhãn hoặc completion.

Giữ dữ liệu/prompt/protocol ViCTSD hiện có: `Title [SEP] Comment`, mapping 2 lớp toxicity và 10 topic; Vistral-7B-Chat, LoRA **r=8 / lora_alpha=16 / dropout=0.05**, độ dài **1024**, batch **6**, accumulation **4**, **3 epochs**, LR **1e-4**, eval/save mỗi **200** bước, seed **42** trước khởi tạo LoRA. `lora_alpha=16` vẫn là hệ số adapter, độc lập với class alpha đã tắt.

Notebook chạy lại baseline trên Colab một GPU hỗ trợ bf16. Đầu ra riêng `victsd_base_ce_r8_len1024_seed42`; không ghi đè, không tự resume. Đây là bản tái tạo baseline theo protocol hiện có, không khẳng định khôi phục chính xác điểm số lịch sử khi thiếu manifest/checkpoint của run cũ.


## 1. Cài thư viện và kết nối

Chạy ô cài đặt một lần. Nếu đã import thư viện trước khi cài, khởi động lại phiên Python rồi chạy từ ô kết nối. Không cài vLLM trong phiên huấn luyện.


In [ ]:
# Bộ phiên bản huấn luyện theo notebook UIT-VSFC tham khảo; không cần vLLM.
%pip install -q transformers==4.46.3 peft==0.10.0 trl==0.8.6 accelerate==0.29.3 datasets==2.21.0 pandas scikit-learn tensorboard


In [ ]:
from pathlib import Path
from google.colab import drive
from huggingface_hub import notebook_login

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

# Đăng nhập tương tác nếu cần; không lưu token trong notebook.
notebook_login()


## 2. Cấu hình baseline

Chỉnh `DATA_DIR` nếu CSV ở nơi khác. Mặc định dùng `ViCTSD_train.csv` và `ViCTSD_valid.csv` trên Drive, cùng nguồn với các bản ViCTSD đã sửa. `SEED=42`; tên run tự thêm seed. Dữ liệu test chỉ là đường dẫn tham chiếu, không được tải khi train.


In [ ]:
import os
import time
import json
import re
import shutil
import hashlib
import math
import importlib.metadata
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, TrainingArguments,
    EarlyStoppingCallback, DataCollatorWithPadding, set_seed,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# Baseline CE: không completion masking, không class alpha hoặc focal.
DATA_DIR = '/content/drive/MyDrive/label-focused-tuning/victsd'
TRAIN_CSV = f'{DATA_DIR}/ViCTSD_train.csv'
VAL_CSV = f'{DATA_DIR}/ViCTSD_valid.csv'
TEST_CSV = f'{DATA_DIR}/ViCTSD_test.csv'  # Chỉ để tham chiếu, không tải khi train.
SEED = 42
RUN_NAME = f'victsd_base_ce_r8_len1024_seed{SEED}'
SOURCE_NOTEBOOKS_SHA256 = {'victsd_mask_alpha_ce_no_focal_r8_len1024.ipynb': '6c79e0eb927652368c447e7fb2fb5013e107a3ef2e809b6eafb0f8f9de4842b8'}
OUTPUT_DIR = f'{DATA_DIR}/{RUN_NAME}'

USE_COMPLETION_MASK = False
USE_ALPHA = False
FOCAL_GAMMA = 0.0
LOSS_NAME = 'base_ce_full_sequence'
MODEL_ID = 'Viet-Mistral/Vistral-7B-Chat'
SEPARATOR = ' [SEP] '

LORA_R = 8
LORA_ALPHA = 16  # Hệ số LoRA, độc lập với class alpha của loss.
LORA_DROPOUT = 0.05
MAX_LENGTH = 1024
BATCH_SIZE = 6
GRAD_ACCUM = 4
NUM_EPOCHS = 3
LEARNING_RATE = 1e-4
EVAL_STEPS = 200

if USE_COMPLETION_MASK or USE_ALPHA or FOCAL_GAMMA != 0.0:
    raise ValueError('Baseline này yêu cầu completion mask=False, class alpha=False và gamma=0.')

print(f'PyTorch: {torch.__version__}; CUDA: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('Notebook huấn luyện yêu cầu GPU CUDA.')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('Cấu hình gốc dùng bf16; hãy chọn GPU hỗ trợ bf16 (ví dụ A100).')
if torch.cuda.device_count() != 1:
    raise RuntimeError('Hai notebook được cấu hình cho một GPU; hãy dùng cùng cấu hình cho các ablation.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

if Path(OUTPUT_DIR).exists() and (not Path(OUTPUT_DIR).is_dir() or any(Path(OUTPUT_DIR).iterdir())):
    raise FileExistsError(f'{OUTPUT_DIR} không rỗng. Đổi RUN_NAME để lưu một lần chạy mới.')
for path in (TRAIN_CSV, VAL_CSV):
    if not Path(path).is_file():
        raise FileNotFoundError(path)

# Đặt seed trước khởi tạo LoRA ở cả hai notebook.
set_seed(SEED)
print(f'Objective: {LOSS_NAME}; prompt + response CE; padding-only mask')
print(f'Seed={SEED}; output={OUTPUT_DIR}')
print(f'LoRA r={LORA_R}, lora_alpha={LORA_ALPHA}; effective batch={BATCH_SIZE * GRAD_ACCUM}')


## 3. Nhãn, prompt và dữ liệu

Lọc train bằng độ dài của toàn prompt gồm JSON. Giữ đủ validation và dừng nếu vượt max length, tránh mất câu trả lời. Tokenize với `add_special_tokens=False` ở cả bước lọc lẫn trainer vì chat template đã có special tokens. Không dùng test để tính alpha/chọn checkpoint.


In [ ]:
SENTIMENT_MAP = {0: 'Non-toxic', 1: 'Toxic'}
SENTIMENT_LABELS = ['Non-toxic', 'Toxic']
TOPIC_LABELS = [
    'KinhDoanh', 'OtoXemay', 'TheGioi', 'TheThao', 'KhoaHoc',
    'ThoiSu', 'PhapLuat', 'GiaoDuc', 'SucKhoe', 'GiaiTri',
]

SYSTEM_PROMPT = (
    'Thực hiện đồng thời hai nhiệm vụ:\n'
    '(1) Sentiment: Non-toxic hoặc Toxic.\n'
    '(2) Topic: KinhDoanh, OtoXemay, TheGioi, TheThao, KhoaHoc, '
    'ThoiSu, PhapLuat, GiaoDuc, SucKhoe, GiaiTri.\n'
    'Chỉ trả về JSON với đúng hai khóa: Sentiment và Topic. Không giải thích thêm.'
)
FEW_SHOT = (
    'Ví dụ:\n'
    'Bình luận: "Biden dẫn Trump 13 điểm [SEP] Cuộc bầu cử ở Mỹ là công khai và công bằng nhất."\n'
    'Output: {"Sentiment": "Non-toxic", "Topic": "TheGioi"}'
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'


def merge_title_comment(df):
    df = df.copy()
    df['Title'] = df['Title'].fillna('').astype(str).str.strip()
    df['Comment'] = df['Comment'].fillna('').astype(str).str.strip()
    df['Text'] = df.apply(
        lambda row: f"{row['Title']}{SEPARATOR}{row['Comment']}" if row['Title'] else row['Comment'],
        axis=1,
    )
    return df


def validate_and_prepare(df, split):
    required = {'Title', 'Comment', 'Toxicity', 'Topic'}
    if not required.issubset(df.columns):
        raise ValueError(f'{split}: thiếu cột {sorted(required - set(df.columns))}')
    if df.empty:
        raise ValueError(f'{split}: dữ liệu rỗng.')
    df = df.copy()
    toxicity = pd.to_numeric(df['Toxicity'], errors='coerce')
    invalid = ~toxicity.isin(SENTIMENT_MAP)
    if invalid.any():
        raise ValueError(f'{split}: Toxicity ngoài 0/1 ở dòng {df.index[invalid].tolist()[:10]}')
    df['Toxicity'] = toxicity.astype('int64')
    df['Topic'] = df['Topic'].fillna('').astype(str).str.strip()
    invalid = ~df['Topic'].isin(TOPIC_LABELS)
    if invalid.any():
        raise ValueError(f'{split}: Topic không hợp lệ ở dòng {df.index[invalid].tolist()[:10]}')
    df = merge_title_comment(df)
    if df['Text'].eq('').any():
        raise ValueError(f'{split}: có dòng rỗng cả Title và Comment.')
    return df


def build_prompt(row):
    target_json = (
        f'{{"Sentiment": "{SENTIMENT_MAP[int(row["Toxicity"])]}", '
        f'"Topic": "{row["Topic"]}"}}'
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'{FEW_SHOT}\n\nBình luận: "{row["Text"]}"'},
        {'role': 'assistant', 'content': target_json},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


df_train = validate_and_prepare(pd.read_csv(TRAIN_CSV), 'train')
df_val = validate_and_prepare(pd.read_csv(VAL_CSV), 'validation')


def prompt_lengths(prompts, batch_size=256):
    lengths = []
    for start in range(0, len(prompts), batch_size):
        encoded = tokenizer(
            prompts[start:start + batch_size], add_special_tokens=False,
            truncation=False, padding=False,
        )
        lengths.extend(len(ids) for ids in encoded['input_ids'])
    return lengths


# Chat template đã chứa special tokens: dùng add_special_tokens=False ở mọi nơi.
all_train_prompts = df_train.apply(build_prompt, axis=1).tolist()
df_train['prompt_len'] = prompt_lengths(all_train_prompts)
train_keep = df_train['prompt_len'].le(MAX_LENGTH).to_numpy()
df_train_filtered = df_train.loc[train_keep].copy()
train_prompts = [text for text, keep in zip(all_train_prompts, train_keep) if keep]
if not train_prompts:
    raise ValueError('Không còn mẫu train sau lọc độ dài.')

val_prompts = df_val.apply(build_prompt, axis=1).tolist()
val_lengths = prompt_lengths(val_prompts)
too_long_val = [i for i, length in enumerate(val_lengths) if length > MAX_LENGTH]
if too_long_val:
    raise ValueError(
        f'{len(too_long_val)} mẫu validation vượt {MAX_LENGTH} token; vị trí {too_long_val[:10]}. '
        'Dừng để tránh cắt mất JSON hoặc tự đổi split. Cần thống nhất xử lý độ dài ở mọi cấu hình đối chiếu.'
    )

train_dataset = Dataset.from_dict({'text': train_prompts})
eval_dataset = Dataset.from_dict({'text': val_prompts})
print(f'Train: {len(df_train)} -> {len(train_dataset)}; loại {len(df_train) - len(train_dataset)} mẫu dài.')
print(f'Validation giữ đủ: {len(eval_dataset)} mẫu.')
print('Phân phối Toxicity sau lọc:\n', df_train_filtered['Toxicity'].value_counts())
print('Phân phối Topic sau lọc:\n', df_train_filtered['Topic'].value_counts())
print('\nPrompt mẫu:\n', train_prompts[0])


## 4. Collator CE toàn chuỗi và kiểm tra dữ liệu

Tạo `labels` bằng bản sao `input_ids`, chỉ đặt `-100` tại `attention_mask == 0`. Không dùng completion-only collator và không tạo `alpha_weights`. Giữ `add_special_tokens=False` vì chat template đã chứa special tokens, và giữ `packing=False` như các run ViCTSD hiện tại.

EOS thật ở vị trí `attention_mask == 1` được giữ trong labels. Không dùng điều kiện `input_ids == pad_token_id` để mask, vì pad và EOS có thể cùng ID.


In [ ]:
class FullSequenceCECollator:
    """Học toàn bộ token có attention=1; chỉ bỏ padding khỏi loss."""
    def __init__(self, tokenizer):
        self.pad = DataCollatorWithPadding(
            tokenizer=tokenizer, padding=True, return_tensors='pt',
        )

    def __call__(self, features):
        if not features:
            raise ValueError('Batch rỗng.')
        # Luôn tái tạo labels, không dùng lại completion mask từ feature cũ.
        model_features = [
            {key: value for key, value in feature.items() if key in {'input_ids', 'attention_mask'}}
            for feature in features
        ]
        batch = self.pad(model_features)
        if batch['input_ids'].shape != batch['attention_mask'].shape:
            raise ValueError('input_ids và attention_mask khác shape.')
        labels = batch['input_ids'].clone()
        labels[batch['attention_mask'] == 0] = -100
        if (labels[:, 1:] != -100).sum(dim=1).eq(0).any().item():
            raise ValueError('Có mẫu không có target CE sau causal shift.')
        batch['labels'] = labels
        return batch


collator = FullSequenceCECollator(tokenizer)

# Chỉ dùng để audit ranh giới prompt/response; KHÔNG dùng ranh giới này để mask loss.
response_template_ids = tokenizer.encode('[/INST]', add_special_tokens=False)


def inspect_serialized_example(ids, expected=None):
    """Kiểm tra target JSON đầy đủ và đếm token prompt/response được học."""
    ids = [int(value) for value in ids]
    width = len(response_template_ids)
    ends = [
        start + width for start in range(len(ids) - width + 1)
        if ids[start:start + width] == response_template_ids
    ]
    if not ends:
        raise ValueError('Không tìm thấy [/INST] trong prompt đã tokenize.')
    response_start = ends[-1]
    if response_start <= 1 or response_start >= len(ids):
        raise ValueError('Prompt/response audit không có đủ token.')
    decoded = tokenizer.decode(
        ids[response_start:], skip_special_tokens=True, clean_up_tokenization_spaces=False,
    ).strip()
    try:
        obj = json.loads(decoded)
    except json.JSONDecodeError as exc:
        raise ValueError('Target JSON không đầy đủ; kiểm tra tokenization và độ dài.') from exc
    if not isinstance(obj, dict) or set(obj) != {'Sentiment', 'Topic'}:
        raise ValueError(f'Target JSON sai khóa: {obj}')
    if obj['Sentiment'] not in SENTIMENT_LABELS or obj['Topic'] not in TOPIC_LABELS:
        raise ValueError(f'Nhãn ngoài mapping: {obj}')
    if expected is not None and obj != expected:
        raise ValueError(f'Target audit sai: {obj} != {expected}')
    return {
        'Sentiment': obj['Sentiment'], 'Topic': obj['Topic'],
        'sequence_tokens': len(ids),
        'shifted_prompt_targets': response_start - 1,
        'shifted_response_targets': len(ids) - response_start,
        'shifted_total_targets': len(ids) - 1,
    }


In [ ]:
# Kiểm tra đủ 20 cặp: prompt và response đều còn labels, EOS thật không bị bỏ.
ce_audit = []
audit_features = []
probe_source = df_train_filtered.loc[df_train_filtered['prompt_len'].idxmin()].copy()
for sent, topic in product(SENTIMENT_LABELS, TOPIC_LABELS):
    probe = probe_source.copy()
    probe['Toxicity'] = next(key for key, value in SENTIMENT_MAP.items() if value == sent)
    probe['Topic'] = topic
    feature = tokenizer(build_prompt(probe), add_special_tokens=False, truncation=False)
    if len(feature['input_ids']) > MAX_LENGTH:
        raise ValueError('Prompt audit vượt MAX_LENGTH.')
    audit_features.append(feature)
    batch = collator([feature])
    keep = batch['attention_mask'].bool()
    if not torch.equal(batch['labels'][keep], batch['input_ids'][keep]):
        raise ValueError('Token prompt/response thật bị mask hoặc thay đổi label.')
    if 'alpha_weights' in batch:
        raise ValueError('Baseline không được có alpha_weights.')
    record = inspect_serialized_example(feature['input_ids'], {'Sentiment': sent, 'Topic': topic})
    if int((batch['labels'][:, 1:] != -100).sum()) != record['shifted_total_targets']:
        raise ValueError('Số target được học không khớp toàn chuỗi sau shift.')
    ce_audit.append(record)

mixed = collator(audit_features)
for row, feature in enumerate(audit_features):
    length = len(feature['input_ids'])
    if not torch.equal(mixed['labels'][row, :length], torch.tensor(feature['input_ids'])):
        raise ValueError('Labels thật bị đổi khi ghép batch có padding.')
    if (mixed['labels'][row, length:] != -100).any().item():
        raise ValueError('Padding chưa được bỏ khỏi loss.')
    real_eos = mixed['input_ids'][row].eq(tokenizer.eos_token_id) & mixed['attention_mask'][row].bool()
    if not torch.equal(mixed['labels'][row, real_eos], mixed['input_ids'][row, real_eos]):
        raise ValueError('EOS thật đã bị mask nhầm thành padding.')

# Kiểm tra toàn bộ train/validation trước khi tải model 7B, không đọc test.
preflight_counts = {}
for split_name, dataset in (('train', train_dataset), ('validation', eval_dataset)):
    checked_count = 0
    for start in range(0, len(dataset), 64):
        texts = dataset[start:start + 64]['text']
        encoded = tokenizer(texts, add_special_tokens=False, truncation=False, padding=False)
        features = [{key: values[i] for key, values in encoded.items()} for i in range(len(texts))]
        batch = collator(features)
        active = batch['attention_mask'].bool()
        if not torch.equal(batch['labels'][active], batch['input_ids'][active]):
            raise ValueError(f'{split_name}: labels thật không khớp input_ids.')
        if batch['labels'][~active].ne(-100).any().item():
            raise ValueError(f'{split_name}: padding vẫn được tính loss.')
        for feature in features:
            if len(feature['input_ids']) > MAX_LENGTH:
                raise ValueError(f'{split_name}: còn mẫu vượt MAX_LENGTH sau tiền xử lý.')
            inspect_serialized_example(feature['input_ids'])
        checked_count += len(features)
    preflight_counts[split_name] = checked_count
    print(f'{split_name}: đã kiểm tra toàn chuỗi CE/JSON cho {checked_count} mẫu.')

assert len(ce_audit) == len(SENTIMENT_LABELS) * len(TOPIC_LABELS) == 20
print(pd.DataFrame(ce_audit).head().to_string(index=False))
print('✅ Đủ 20/20 cặp: prompt + response đều học; chỉ padding bị mask; không alpha/focal.')


## 5. Cross-entropy chuẩn và kiểm tra số học

Lấy mean trên mọi target không phải padding sau causal shift. Trainer tính CE một lần, không nhân alpha hoặc focal. Giữ chuẩn hóa theo microbatch như các trainer ViCTSD tham chiếu.


In [ ]:
def full_sequence_ce_loss(logits, labels):
    """CE trên toàn chuỗi sau shift, bỏ đúng các labels=-100 của padding."""
    if logits.shape[:2] != labels.shape:
        raise ValueError('Shape logits và labels không khớp.')
    shifted_labels = labels[..., 1:].to(logits.device)
    keep = shifted_labels != -100
    if not keep.any().item():
        raise ValueError('Batch không có target CE sau causal shift.')
    selected_logits = logits[..., :-1, :][keep].float()
    return F.cross_entropy(selected_logits, shifted_labels[keep], reduction='mean')


class BaseCETrainer(SFTTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        model_inputs = dict(inputs)
        if 'alpha_weights' in model_inputs:
            raise ValueError('Baseline CE không nhận alpha_weights.')
        labels = model_inputs.pop('labels')
        outputs = model(**model_inputs)
        loss = full_sequence_ce_loss(outputs.logits, labels)
        outputs['loss'] = loss
        return (loss, outputs) if return_outputs else loss


def check_base_ce_math():
    logits = torch.linspace(-2.0, 2.0, 2 * 5 * 7).reshape(2, 5, 7).requires_grad_()
    labels = torch.tensor([[0, 1, 2, 3, 4], [0, 5, 6, -100, -100]])
    expected_terms = []
    for row in range(labels.size(0)):
        for target_pos in range(1, labels.size(1)):
            target = int(labels[row, target_pos])
            if target != -100:
                expected_terms.append(-F.log_softmax(logits[row, target_pos - 1].float(), dim=-1)[target])
    loss = full_sequence_ce_loss(logits, labels)
    torch.testing.assert_close(loss, torch.stack(expected_terms).mean())
    standard = F.cross_entropy(
        logits[:, :-1, :].reshape(-1, 7).float(), labels[:, 1:].reshape(-1), ignore_index=-100,
    )
    torch.testing.assert_close(loss, standard)

    # Logit dự đoán token đầu thuộc prompt phải ảnh hưởng đến loss của baseline.
    changed_prompt = logits.detach().clone()
    changed_prompt[0, 0, 1] += 3.0
    if torch.isclose(full_sequence_ce_loss(changed_prompt, labels), loss).item():
        raise AssertionError('Prompt không ảnh hưởng loss; kiểm tra completion mask còn sót.')
    # Vị trí dự đoán padding và logit cuối không được ảnh hưởng loss.
    changed_padding = logits.detach().clone()
    changed_padding[1, 2:, :] = 100.0
    torch.testing.assert_close(full_sequence_ce_loss(changed_padding, labels), loss)
    loss.backward()
    if not torch.isfinite(logits.grad).all().item():
        raise AssertionError('Gradient không hữu hạn.')
    if logits.grad[0, 0].abs().sum().item() == 0:
        raise AssertionError('Prompt không có gradient.')
    if logits.grad[1, 2:].abs().sum().item() != 0:
        raise AssertionError('Padding hoặc logit cuối vẫn có gradient.')
    try:
        full_sequence_ce_loss(logits, torch.full_like(labels, -100))
    except ValueError:
        pass
    else:
        raise AssertionError('Batch rỗng target phải báo lỗi.')
    print('✅ CE khớp công thức chuẩn; prompt có loss/gradient, padding được bỏ đúng, causal shift đúng.')


check_base_ce_math()


## 6. Model, LoRA và trainer baseline

Giữ cấu hình ViCTSD, seed trước LoRA và lịch train 3 epochs. `label_names=['labels']` bảo đảm evaluation dùng cùng CE toàn chuỗi. Best checkpoint được chọn theo validation loss của baseline này; không so trực tiếp giá trị loss với các objective masking/alpha.


In [ ]:
print('Đang tải Vistral-7B bằng bf16, không quantize...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Đặt lại ngay trước LoRA để các kiểm tra trước đó không làm lệch RNG giữa hai run.
set_seed(SEED)
peft_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none', task_type='CAUSAL_LM',
)
model_with_lora = get_peft_model(model, peft_config)
model_with_lora.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    optim='adamw_torch_fused', learning_rate=LEARNING_RATE,
    weight_decay=0.01, max_grad_norm=1.0,
    lr_scheduler_type='cosine', warmup_ratio=0.05,
    bf16=True, fp16=False, tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    evaluation_strategy='steps', eval_steps=EVAL_STEPS,
    save_strategy='steps', save_steps=EVAL_STEPS, save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss', greater_is_better=False,
    logging_steps=10, report_to='tensorboard',
    group_by_length=True,
    dataloader_num_workers=2, dataloader_pin_memory=True,
    seed=SEED, data_seed=SEED,
    label_names=['labels'], prediction_loss_only=True,
)

trainer = BaseCETrainer(
    model=model_with_lora,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    tokenizer=tokenizer, args=training_args,
    dataset_text_field='text', max_seq_length=MAX_LENGTH, packing=False,
    dataset_kwargs={'add_special_tokens': False},
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4, early_stopping_threshold=0.001)],
)

# Đối chiếu tokenizer của SFTTrainer với tokenizer dùng lọc/audit.
for dataset_name, actual_dataset, source_prompts in (
    ('train', trainer.train_dataset, train_prompts),
    ('validation', trainer.eval_dataset, val_prompts),
):
    for index in sorted({0, len(actual_dataset) - 1}):
        expected_ids = tokenizer(source_prompts[index], add_special_tokens=False)['input_ids']
        if actual_dataset[index]['input_ids'] != expected_ids:
            raise ValueError(f'{dataset_name}: tokenization của trainer khác bước tiền xử lý.')

steps_per_epoch = math.ceil(math.ceil(len(train_dataset) / BATCH_SIZE) / GRAD_ACCUM)
estimated_steps = steps_per_epoch * NUM_EPOCHS
if estimated_steps < EVAL_STEPS:
    raise ValueError('Không đủ bước để có validation checkpoint; thống nhất eval/save_steps ở mọi cấu hình.')
print(f'Số bước dự kiến khoảng {estimated_steps}; eval/save mỗi {EVAL_STEPS} bước.')
print(f'VRAM allocated trước train: {torch.cuda.memory_allocated() / 1e9:.1f} GB')


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def frame_sha256(frame):
    payload = frame[['Title', 'Comment', 'Text', 'Toxicity', 'Topic']].to_json(
        orient='split', force_ascii=False,
    )
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


manifest = {
    'run_name': RUN_NAME, 'status': 'ready', 'dataset': 'ViCTSD', 'model_id': MODEL_ID,
    'notebook': 'victsd_base_ce_r8_len1024.ipynb',
    'reference_notebook': 'victsd_mask_alpha_ce_no_focal_r8_len1024.ipynb',
    'source_notebooks_sha256': SOURCE_NOTEBOOKS_SHA256,
    'model_resolved_revision': getattr(model.config, '_commit_hash', None),
    'tokenizer_resolved_revision': tokenizer.init_kwargs.get('_commit_hash'),
    'tokenizer_backend_sha256': (
        hashlib.sha256(tokenizer.backend_tokenizer.to_str().encode('utf-8')).hexdigest()
        if getattr(tokenizer, 'is_fast', False) else None
    ),
    'loss': {
        'name': LOSS_NAME, 'completion_mask': False,
        'scope': 'prompt_and_response_full_sequence',
        'gamma': 0.0, 'focal_modulation': False, 'alpha_balancing': False,
        'reduction': 'mean_over_nonpadding_shifted_full_sequence_tokens',
        'padding_mask_source': 'attention_mask_eq_zero',
        'real_eos_in_loss': True,
    },
    'lora': {'r': LORA_R, 'lora_alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT},
    'max_length': MAX_LENGTH, 'seed_before_lora': SEED,
    'add_special_tokens': False, 'packing': False,
    'train_rows_raw': len(df_train), 'train_rows_filtered': len(df_train_filtered),
    'train_rows_removed_for_length': len(df_train) - len(df_train_filtered),
    'validation_rows': len(df_val),
    'train_filtered_sha256': frame_sha256(df_train_filtered),
    'validation_sha256': frame_sha256(df_val),
    'data_files': {'train': TRAIN_CSV, 'validation': VAL_CSV},
    'data_file_sha256': {'train': file_sha256(TRAIN_CSV), 'validation': file_sha256(VAL_CSV)},
    'test_used_during_training': False,
    'ce_audit_20_pairs': ce_audit,
    'preflight_counts': preflight_counts,
    'system_prompt': SYSTEM_PROMPT, 'few_shot': FEW_SHOT,
    'response_template_ids_audit_only': response_template_ids,
    'training_arguments': json.loads(training_args.to_json_string()),
    'checkpoint_selection': 'minimum_validation_loss_of_this_objective',
    'runtime_versions': {
        name: importlib.metadata.version(name)
        for name in ['torch', 'transformers', 'tokenizers', 'peft', 'trl', 'accelerate', 'datasets', 'numpy', 'pandas', 'scikit-learn']
    },
}


def save_manifest():
    with open(Path(OUTPUT_DIR) / 'run_manifest.json', 'w', encoding='utf-8') as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)


Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
save_manifest()
print('Đã lưu cấu hình baseline CE, hash dữ liệu và audit tại:', OUTPUT_DIR)


## 7. Huấn luyện và lưu

Ô tiếp theo bắt đầu train. `best_model` là adapter đã chọn theo validation loss. `second_best_model` chỉ được tạo nếu checkpoint hạng hai còn trên đĩa. `last_checkpoint_model` là checkpoint lưu cuối theo lịch 200 bước, có thể sớm hơn bước kết thúc train; bước thực tế được ghi vào manifest.


In [ ]:
print(f'Bắt đầu huấn luyện {RUN_NAME}...')
manifest['status'] = 'training'
save_manifest()
start_time = time.time()
try:
    train_result = trainer.train()
except Exception as error:
    manifest.update(status='failed', error_type=type(error).__name__, error=str(error))
    save_manifest()
    raise
total_time = time.time() - start_time
trainer.save_state()

if trainer.state.best_model_checkpoint is None:
    raise RuntimeError('Chưa có best checkpoint theo validation; không đặt tên best_model cho model chưa được chọn.')

# load_best_model_at_end=True: adapter hiện tại là best theo objective của run này.
best_dir = Path(OUTPUT_DIR) / 'best_model'
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(best_dir)
saved_models = {'best_model': str(best_dir)}

# Mỗi step chỉ xếp hạng một lần. Second-best có thể đã bị checkpoint rotation xóa.
eval_by_step = {
    int(log['step']): log for log in trainer.state.log_history
    if 'eval_loss' in log and 'step' in log and math.isfinite(float(log['eval_loss']))
}
ranked_logs = sorted(eval_by_step.values(), key=lambda log: (log['eval_loss'], log['step']))
if len(ranked_logs) >= 2:
    second_step = int(ranked_logs[1]['step'])
    second_src = Path(OUTPUT_DIR) / f'checkpoint-{second_step}'
    if second_src.is_dir():
        second_dir = Path(OUTPUT_DIR) / 'second_best_model'
        shutil.copytree(second_src, second_dir)
        tokenizer.save_pretrained(second_dir)
        saved_models['second_best_model'] = str(second_dir)
    else:
        print(f'Checkpoint hạng hai (step {second_step}) đã bị rotation xóa; không tạo second_best_model.')

available_steps = [
    int(path.name.split('-')[-1]) for path in Path(OUTPUT_DIR).iterdir()
    if path.is_dir() and re.fullmatch(r'checkpoint-\d+', path.name)
]
last_saved_step = max(available_steps) if available_steps else None
if last_saved_step is not None:
    last_src = Path(OUTPUT_DIR) / f'checkpoint-{last_saved_step}'
    last_dir = Path(OUTPUT_DIR) / 'last_checkpoint_model'
    shutil.copytree(last_src, last_dir)
    tokenizer.save_pretrained(last_dir)
    saved_models['last_saved_checkpoint'] = str(last_dir)

manifest.update({
    'status': 'completed', 'train_seconds': total_time,
    'final_global_step': int(trainer.state.global_step),
    'best_model_checkpoint': trainer.state.best_model_checkpoint,
    'best_validation_loss': trainer.state.best_metric,
    'last_saved_checkpoint_step': last_saved_step,
    'saved_models': saved_models,
    'eval_logs_sorted_by_objective_loss': ranked_logs,
    'train_metrics': train_result.metrics,
})
save_manifest()

print(f'Hoàn tất sau {total_time / 3600:.2f} giờ.')
print(f'Best checkpoint: {trainer.state.best_model_checkpoint}')
print(f'Best validation objective loss: {trainer.state.best_metric}')
print(f'Bước kết thúc: {trainer.state.global_step}; checkpoint lưu cuối: {last_saved_step}')
for name, folder in saved_models.items():
    print(f'{name}: {folder}')
print('Đã lưu trainer_state.json và run_manifest.json.')


## Lấy lại kết quả baseline CE

Sau train, dùng adapter trong `OUTPUT_DIR + "/best_model"`. Chạy notebook test ViCTSD dùng chung `victsd_test_ablation_fixed.ipynb`, đổi `ADAPTER_PATH` sang adapter mới. Giữ cùng dữ liệu test, prompt, decoder, parser và tiêu chí chọn checkpoint để đối chiếu các nhánh khác. Code test sẽ lưu dự đoán, raw outputs, metrics và manifest đánh giá vào thư mục kết quả riêng.

Nếu vẫn còn adapter baseline CE cũ thì chỉ cần đánh giá lại adapter đó bằng notebook test; không cần train lại để lấy lại báo cáo đã mất. Notebook hiện tại dùng khi cần tái huấn luyện baseline theo protocol đang có; không bảo đảm trùng kết quả lịch sử khi thiếu trạng thái/manifest của run cũ.

Đặc điểm baseline trong file này: CE trên **prompt + response**, không completion masking/class alpha/focal; chỉ bỏ padding và giữ EOS thật. `lora_alpha=16` vẫn giữ nguyên vì đó là hệ số adapter. Manifest ghi rõ quy ước loss, seed, hash dữ liệu/tokenizer, cấu hình và adapter đã lưu.

Checkpoint giữ cách lưu ViCTSD: `best_model`, `second_best_model` nếu checkpoint còn tồn tại và `last_checkpoint_model` theo checkpoint lưu cuối. CSV thực tế và huấn luyện model 7B chưa được chạy trong lúc tạo file; audit sẽ chạy trên Colab trước khi tải model.
